In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

#download data from kaggle
!kaggle competitions download -c hms-harmful-brain-activity-classification


!mkdir -p dataset
!unzip -q hms-harmful-brain-activity-classification.zip -d dataset/

!rm hms-harmful-brain-activity-classification.zip

!echo "loading success!"

100% 18.4G/18.4G [01:04<00:00, 534MB/s]
100% 18.4G/18.4G [01:04<00:00, 309MB/s]
loading success!


In [6]:
#only efficient
import torch
import torch.nn as nn
import timm

class HMSModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=6, dropout_rate=0.5):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=True)
        # Replace the classifier head for our specific number of classes
        in_features = self.model.classifier.in_features
        self.model.classifier = nn.Identity() # Remove original classifier

        self.dropout = nn.Dropout(dropout_rate) # Add dropout layer
        self.fc = nn.Linear(in_features, num_classes) # New fully connected layer

    def forward(self, x):
        # Pass through the base model (without its original classifier)
        x = self.model(x)
        x = self.dropout(x) # Apply dropout
        x = self.fc(x) # Pass through new fully connected layer
        return x

In [4]:
!pip install torchinfo

In [7]:
from torchinfo import summary

model = HMSModel(model_name='efficientnet_b0', num_classes=6, dropout_rate=0.5)

model_summary = summary(
    model,
    input_size=(32, 3, 256, 256),
    col_names=["output_size", "num_params"],
    depth=2,
    verbose=1
)

print(model_summary)

Layer (type:depth-idx)                             Output Shape              Param #
HMSModel                                           [32, 6]                   --
├─EfficientNet: 1-1                                [32, 1280]                --
│    └─Conv2d: 2-1                                 [32, 32, 128, 128]        864
│    └─BatchNormAct2d: 2-2                         [32, 32, 128, 128]        64
│    └─Sequential: 2-3                             [32, 320, 8, 8]           3,594,460
│    └─Conv2d: 2-4                                 [32, 1280, 8, 8]          409,600
│    └─BatchNormAct2d: 2-5                         [32, 1280, 8, 8]          2,560
│    └─SelectAdaptivePool2d: 2-6                   [32, 1280]                --
│    └─Identity: 2-7                               [32, 1280]                --
├─Dropout: 1-2                                     [32, 1280]                --
├─Linear: 1-3                                      [32, 6]                   7,686
Total params: 4,

In [2]:
#with transformer
import torch
import torch.nn as nn
import timm

class HMSModel(nn.Module):
    def __init__(
        self,
        model_name='efficientnet_b0',
        num_classes=6,
        dropout_rate=0.5,
        use_transformer=True,
        d_model=256,
        nhead=4,
        num_layers=2,
        transformer_dropout=0.1,
        max_tokens=128
    ):
        super().__init__()

        # CNN backbone
        self.backbone = timm.create_model(model_name, pretrained=True)
        self.feature_dim = self.backbone.num_features
        self.use_transformer = use_transformer

        if self.use_transformer:
            # Project CNN channels -> transformer hidden dim
            self.proj = nn.Linear(self.feature_dim, d_model)

            # Learnable positional embedding
            self.pos_embed = nn.Parameter(torch.zeros(1, max_tokens, d_model))

            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model * 4,
                dropout=transformer_dropout,
                activation='gelu',
                batch_first=True,
                norm_first=True
            )
            self.transformer = nn.TransformerEncoder(
                encoder_layer,
                num_layers=num_layers
            )
            self.norm = nn.LayerNorm(d_model)
            head_in_features = d_model

            self._init_transformer_weights()
        else:
            head_in_features = self.feature_dim

        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(head_in_features, num_classes)

    def _init_transformer_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        nn.init.xavier_uniform_(self.proj.weight)
        nn.init.zeros_(self.proj.bias)

        for m in self.transformer.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # EfficientNet feature map
        # Expected shape for 256x256 input is typically [B, C, H, W], e.g. [B, 1280, 8, 8]
        feat = self.backbone.forward_features(x)

        if feat.ndim != 4:
            raise ValueError(f"Expected 4D feature map from backbone, got shape: {feat.shape}")

        # [B, C, H, W] -> [B, H*W, C]
        feat = feat.flatten(2).transpose(1, 2)

        if self.use_transformer:
            feat = self.proj(feat)   # [B, N, d_model]

            n_tokens = feat.size(1)
            if n_tokens > self.pos_embed.size(1):
                raise ValueError(
                    f"Number of tokens ({n_tokens}) exceeds max_tokens ({self.pos_embed.size(1)}). "
                    "Increase max_tokens."
                )

            feat = feat + self.pos_embed[:, :n_tokens, :]
            feat = self.transformer(feat)
            feat = self.norm(feat)

        # Global average over tokens
        feat = feat.mean(dim=1)

        feat = self.dropout(feat)
        out = self.fc(feat)
        return out

In [ ]:
import torch
import pandas as pd
import numpy as np
import cv2
import os

class HMSDataset(torch.utils.data.Dataset):
    def __init__(self, df, data_path, mode='train', augment=False):
        self.df = df
        self.data_path = data_path
        self.mode = mode
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        # Load 10-minute Kaggle Spectrogram (400 frequencies x 300 time steps)
        spec_id = row['spectrogram_id']
        path = os.path.join(self.data_path, 'train_spectrograms', f"{spec_id}.parquet")

        # Read parquet and fill NaNs
        # Columns: [time, LL_freqs..., RL_freqs..., LP_freqs..., RP_freqs...]
        data = pd.read_parquet(path).fillna(0).values[:, 1:].T # (400, 300)

        # --- SpecAugment: Time and Frequency Masking ---
        if self.augment:
            # Frequency Masking
            num_masks = 1
            F = 30 # Max width of the mask
            for _ in range(num_masks):
                f = np.random.randint(0, F)
                f0 = np.random.randint(0, 400 - f)
                data[f0:f0+f, :] = 0

            # Time Masking
            T = 30 # Max width of the mask
            for _ in range(num_masks):
                t = np.random.randint(0, T)
                t0 = np.random.randint(0, 300 - t)
                data[:, t0:t0+t] = 0

        # Split into 4 regions (100 rows each)
        # LL: 0-100, RL: 100-200, LP: 200-300, RP: 300-400
        # We tile them into a 2x2 grid for the CNN
        img = np.zeros((256, 256), dtype='float32')
        for i in range(4):
            region = data[i*100:(i+1)*100, :]
            # Log transform and normalize
            region = np.log1p(region)
            region = (region - np.mean(region)) / (np.std(region) + 1e-6)

            # Resize region to 128x128 and place in quadrant
            res_region = cv2.resize(region, (128, 128))
            r, c = (i // 2) * 128, (i % 2) * 128
            img[r:r+128, c:c+128] = res_region

        # Stack to 3 channels for EfficientNet
        img_3c = np.stack([img, img, img], axis=0) # (3, 256, 256)

        if self.mode == 'train':
            labels = row[['seizure_vote', 'lpd_vote', 'gpd_vote',
                          'lrda_vote', 'grda_vote', 'other_vote']].values.astype('float32')
            labels = labels / labels.sum() # Normalize to probabilities
            return torch.tensor(img_3c), torch.tensor(labels)

        return torch.tensor(img_3c)

In [ ]:
#mixup
import torch
import numpy as np

def mixup_data(x, y, alpha=0.2):
    '''Applies Mixup augmentation to a batch of data.
    Args:
        x (torch.Tensor): Input data (e.g., images).
        y (torch.Tensor): Labels corresponding to the input data.
        alpha (float): The alpha parameter for the Beta distribution. Default is 0.2.
    Returns:
        mixed_x (torch.Tensor): The mixed input data.
        y_a (torch.Tensor): The first set of original labels.
        y_b (torch.Tensor): The second set of original labels.
        lambda_ (float): The mixing coefficient.
    '''
    if alpha > 0:
        lambda_ = np.random.beta(alpha, alpha)
    else:
        lambda_ = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lambda_ * x + (1 - lambda_) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lambda_

print("Mixup data function defined.")

Mixup data function defined.


In [ ]:
#model training with only efficientnet
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import CosineAnnealingLR # Changed import statement

# Settings
DATA_PATH = "./dataset"
BATCH_SIZE = 32
EPOCHS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHT_DECAY = 1e-3

def train():
    # 1. Read CSV
    df = pd.read_csv("/content/confident_train.csv")
    print(f"Total rows in CSV: {len(df)}")
    print(f"Unique patients: {df['patient_id'].nunique()}")
    print(f"Unique eeg_id: {df['eeg_id'].nunique()}")

    # 2. Patient-level split
    train_patients, val_patients = train_test_split(
        df["patient_id"].unique(),
        test_size=0.1,
        random_state=42
    )

    train_df = df[df["patient_id"].isin(train_patients)].reset_index(drop=True)
    val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)

    train_df = train_df.groupby('eeg_id').head(1).reset_index(drop=True)
    val_df = val_df.groupby('eeg_id').head(1).reset_index(drop=True)

    print(f"Train rows: {len(train_df)}")
    print(f"Val rows: {len(val_df)}")
    print(f"Train unique eeg_id: {train_df['eeg_id'].nunique()}")
    print(f"Val unique eeg_id: {val_df['eeg_id'].nunique()}")

    if len(train_df) == 0 or len(val_df) == 0:
        raise ValueError("train_df or val_df is empty. Check your split or CSV.")

    # 3. Dataset / Dataloader
    # Enable augmentation for training set
    train_dataset = HMSDataset(train_df, DATA_PATH, mode="train", augment=True)
    val_dataset = HMSDataset(val_df, DATA_PATH, mode="train", augment=False)

    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Val dataset size: {len(val_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Train loader batches: {len(train_loader)}")
    print(f"Val loader batches: {len(val_loader)}")

    # 4. Model / optimizer / loss
    model = HMSModel().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=WEIGHT_DECAY)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    #scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.KLDivLoss(reduction="batchmean")

    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0
    early_stop_patience = 5

    print(f"Starting training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # ----- Train -----
        model.train()
        total_train_loss = 0.0

        for step, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True) # MODIFIED LINE

            mixed_x, y_a, y_b, lambda_ = mixup_data(imgs, labels, alpha=0.4)

            optimizer.zero_grad()
            output = model(mixed_x)

            loss = (
                lambda_ * criterion(torch.log_softmax(output, dim=1), y_a)
                + (1 - lambda_) * criterion(torch.log_softmax(output, dim=1), y_b)
            )

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)

        # ----- Validation -----
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                output = model(imgs)
                loss = criterion(torch.log_softmax(output, dim=1), labels)
                total_val_loss += loss.item()

        val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        scheduler.step(val_loss)

        # save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), "/content/hms_efficientnet_b0_best.pth")
            print(f"Saved best model at epoch {best_epoch} with val loss {best_val_loss:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= early_stop_patience:
            print("Early stopping triggered.")
            break

    # also save last model just in case
    torch.save(model.state_dict(), "/content/hms_efficientnet_b0_last.pth")
    print(f"Training complete. Best val loss: {best_val_loss:.4f} at epoch {best_epoch}")

    # --- Testing ---
    # Load best model for testing
    model.load_state_dict(torch.load("/content/hms_efficientnet_b0_best.pth"))
    model.eval()

    # Setup Test Data
    test_df = pd.read_csv('/content/confident_test.csv')
    train_specs_dir = os.path.join(DATA_PATH, 'train_spectrograms')
    if os.path.exists(train_specs_dir):
        existing_specs = set(int(f.split('.')[0]) for f in os.listdir(train_specs_dir) if f.endswith('.parquet'))
        test_df = test_df[test_df['spectrogram_id'].isin(existing_specs)].reset_index(drop=True)

        if len(test_df) > 0:
            test_dataset = HMSDataset(test_df, DATA_PATH, mode='train', augment=False) # No Augment for Test!
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

            total_test_loss = 0
            with torch.no_grad():
                for imgs, labels in test_loader:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    output = model(imgs)
                    loss = criterion(torch.log_softmax(output, dim=1), labels)
                    total_test_loss += loss.item()
            print(f"Final Test Loss (using best model): {total_test_loss / len(test_loader):.4f}")

if __name__ == "__main__":
    train()

Total rows in CSV: 61102
Unique patients: 1654
Unique eeg_id: 11416
Train rows: 10134
Val rows: 1282
Train unique eeg_id: 10134
Val unique eeg_id: 1282
Train dataset size: 10134
Val dataset size: 1282
Train loader batches: 317
Val loader batches: 41


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Starting training on cuda...
Epoch 1/50 | Train Loss: 1.1724 | Val Loss: 0.9406
Saved best model at epoch 1 with val loss 0.9406
Epoch 2/50 | Train Loss: 0.9534 | Val Loss: 0.9077
Saved best model at epoch 2 with val loss 0.9077
Epoch 3/50 | Train Loss: 0.9034 | Val Loss: 0.7787
Saved best model at epoch 3 with val loss 0.7787
Epoch 4/50 | Train Loss: 0.8420 | Val Loss: 0.7644
Saved best model at epoch 4 with val loss 0.7644
Epoch 5/50 | Train Loss: 0.8018 | Val Loss: 0.7792
Epoch 6/50 | Train Loss: 0.7440 | Val Loss: 0.7622
Saved best model at epoch 6 with val loss 0.7622
Epoch 7/50 | Train Loss: 0.7538 | Val Loss: 0.7905


In [ ]:
#gen csv with best model
import os
import torch
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader

# Ensure HMSModel and HMSDataset are defined (assuming they are in previous cells)
# from your_model_file import HMSModel
# from your_dataset_file import HMSDataset

# Settings (ensure these match your training setup)
DATA_PATH = "./dataset"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")

def generate_predictions():
    # Initialize the model (using the same architecture as the trained model)
    model = HMSModel().to(DEVICE)

    # Load the best trained model weights
    model_path = "/content/hms_efficientnet_b0_best.pth"
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location=DEVICE))
        print(f"Loaded best model from {model_path}")
    else:
        print(f"Error: Model file not found at {model_path}. Please ensure training was completed successfully and the model was saved.")
        return

    model.eval() # Set model to evaluation mode

    # Load test data
    test_df = pd.read_csv('/content/confident_test.csv')

    # Filter test_df to include only spectrograms that actually exist
    train_specs_dir = os.path.join(DATA_PATH, 'train_spectrograms')
    if os.path.exists(train_specs_dir):
        existing_specs = set(int(f.split('.')[0]) for f in os.listdir(train_specs_dir) if f.endswith('.parquet'))
        test_df_filtered = test_df[test_df['spectrogram_id'].isin(existing_specs)].reset_index(drop=True)
    else:
        # Fallback if train_spectrograms directory does not exist
        print("Warning: 'train_spectrograms' directory not found. Proceeding with all spectrogram_ids in confident_test.csv.")
        test_df_filtered = test_df

    if len(test_df_filtered) == 0:
        print("No test data available for prediction after filtering.")
        return

    test_dataset = HMSDataset(test_df_filtered, DATA_PATH, mode='test', augment=False) # Use 'test' mode if your dataset class handles it, otherwise 'train' with augment=False
    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Test dataset size: {len(test_dataset)}")
    print(f"Test loader batches: {len(test_loader)}")

    all_predictions = []
    with torch.no_grad():
        for imgs in test_loader:
            # If your HMSDataset returns (imgs, labels) even in 'test' mode,
            # you might need to adjust this line:
            # imgs = imgs[0] if isinstance(imgs, (list, tuple)) else imgs
            imgs = imgs.to(DEVICE, non_blocking=True)
            output = model(imgs)
            all_predictions.append(torch.softmax(output, dim=1).cpu().numpy())

    # Concatenate all predictions
    all_predictions = np.concatenate(all_predictions, axis=0)

    # Create submission DataFrame
    submission_df = pd.DataFrame(
        all_predictions,
        columns=['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
    )
    # Add eeg_id back for submission mapping
    submission_df['eeg_id'] = test_df_filtered['eeg_id'].values

    # Reorder columns to match expected submission format (eeg_id first)
    submission_df = submission_df[['eeg_id', 'seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']]

    # Save to CSV
    submission_path = '/content/submission.csv'
    submission_df.to_csv(submission_path, index=False)
    print(f"Prediction CSV saved to {submission_path}")
    print(f"First 5 rows of the submission file:\n{submission_df.head()}")

# Run the prediction generation function
generate_predictions()

Using device: cuda
Loaded best model from /content/hms_efficientnet_b0_best.pth
Test dataset size: 6621
Test loader batches: 207
Prediction CSV saved to /content/submission.csv
First 5 rows of the submission file:
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  4245882082      0.139241  0.000455  0.000069   0.007298   0.189517   
1  2857590162      0.062866  0.004923  0.047150   0.049780   0.211486   
2  2857590162      0.062866  0.004923  0.047150   0.049780   0.211486   
3  2857590162      0.062866  0.004923  0.047150   0.049780   0.211486   
4  3523834378      0.769457  0.120843  0.013196   0.017368   0.027266   

   other_vote  
0    0.663419  
1    0.623796  
2    0.623796  
3    0.623796  
4    0.051870  


In [ ]:
#training model with transformer
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim.lr_scheduler import CosineAnnealingLR # Changed import statement

# Settings
DATA_PATH = "./dataset"
BATCH_SIZE = 32
EPOCHS = 40
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHT_DECAY = 5e-2

def train():
    # Read CSV
    df = pd.read_csv("/content/confident_train.csv")
    print(f"Total rows in CSV: {len(df)}")
    print(f"Unique patients: {df['patient_id'].nunique()}")
    print(f"Unique eeg_id: {df['eeg_id'].nunique()}")

    # Patient-level split
    train_patients, val_patients = train_test_split(
        df["patient_id"].unique(),
        test_size=0.1,
        random_state=42
    )

    train_df = df[df["patient_id"].isin(train_patients)].reset_index(drop=True)
    val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)

    train_df = train_df.groupby('eeg_id').head(1).reset_index(drop=True)
    val_df = val_df.groupby('eeg_id').head(1).reset_index(drop=True)

    print(f"Train rows: {len(train_df)}")
    print(f"Val rows: {len(val_df)}")
    print(f"Train unique eeg_id: {train_df['eeg_id'].nunique()}")
    print(f"Val unique eeg_id: {val_df['eeg_id'].nunique()}")

    if len(train_df) == 0 or len(val_df) == 0:
        raise ValueError("train_df or val_df is empty. Check your split or CSV.")

    # 3. Dataset / Dataloader
    # Enable augmentation for training set
    train_dataset = HMSDataset(train_df, DATA_PATH, mode="train", augment=True)
    val_dataset = HMSDataset(val_df, DATA_PATH, mode="train", augment=False)

    print(f"Train dataset size: {len(train_dataset)}")
    print(f"Val dataset size: {len(val_dataset)}")

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    print(f"Train loader batches: {len(train_loader)}")
    print(f"Val loader batches: {len(val_loader)}")

    # 4. Model / optimizer / loss
    #model = HMSModel().to(DEVICE) #without transformer
    model = HMSModel(
      model_name='efficientnet_b0',
      num_classes=6,
      dropout_rate=0.4,
      use_transformer=True,
      d_model=128,
      nhead=4,
      num_layers=1,
      transformer_dropout=0.1,
      max_tokens=128
    ).to(DEVICE)

    #optimizer = torch.optim.Adam(model.parameters(), lr=5e-4) #, weight_decay=WEIGHT_DECAY)
    backbone_params = []
    new_params = []

    for name, param in model.named_parameters():
      if not param.requires_grad:
          continue
      if name.startswith("backbone."):
        backbone_params.append(param)
      else:
        new_params.append(param)

    optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": 2e-4},
        {"params": new_params, "lr": 5e-4},
    ],
    weight_decay=WEIGHT_DECAY
    )
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)
    #scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    criterion = nn.KLDivLoss(reduction="batchmean")


    #=====optional========
    dummy = torch.randn(2, 3, 256, 256).to(DEVICE)
    with torch.no_grad():
      dummy_out = model(dummy)
    print("Dummy output shape:", dummy_out.shape)

    imgs, labels = next(iter(train_loader))
    imgs = imgs.to(DEVICE)
    labels = labels.to(DEVICE)

    with torch.no_grad():
        out = model(imgs)

    print("Batch imgs shape:", imgs.shape)
    print("Batch labels shape:", labels.shape)
    print("Batch output shape:", out.shape)
    print("Labels sum example:", labels[0].sum().item())
    #=====================

    best_val_loss = float("inf")
    best_epoch = -1
    patience_counter = 0
    early_stop_patience = 10

    print(f"Starting training on {DEVICE}...")

    for epoch in range(EPOCHS):
        # ----- Train -----
        model.train()
        total_train_loss = 0.0

        for step, (imgs, labels) in enumerate(train_loader):
            imgs = imgs.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True) # MODIFIED LINE

            mixed_x, y_a, y_b, lambda_ = mixup_data(imgs, labels, alpha=0.4)

            optimizer.zero_grad()
            output = model(mixed_x)

            loss = (
                lambda_ * criterion(torch.log_softmax(output, dim=1), y_a)
                + (1 - lambda_) * criterion(torch.log_softmax(output, dim=1), y_b)
            )

            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        train_loss = total_train_loss / len(train_loader)

        # ----- Validation -----
        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)

                output = model(imgs)
                loss = criterion(torch.log_softmax(output, dim=1), labels)
                total_val_loss += loss.item()

        val_loss = total_val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

        scheduler.step(val_loss)

        # save best
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            patience_counter = 0
            torch.save(model.state_dict(), "/content/hms_effb0_transformer_best.pth")
            print(f"Saved best model at epoch {best_epoch} with val loss {best_val_loss:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= early_stop_patience:
            print("Early stopping triggered.")
            break

    # also save last model just in case
    torch.save(model.state_dict(), "/content/hms_effb0_transformer_last.pth")
    print(f"Training complete. Best val loss: {best_val_loss:.4f} at epoch {best_epoch}")

    # --- Testing ---
    # Load best model for testing
    model.load_state_dict(torch.load("/content/hms_effb0_transformer_best.pth"))
    model.eval()

    # Setup Test Data
    test_df = pd.read_csv('/content/confident_test.csv')
    train_specs_dir = os.path.join(DATA_PATH, 'train_spectrograms')
    if os.path.exists(train_specs_dir):
        existing_specs = set(int(f.split('.')[0]) for f in os.listdir(train_specs_dir) if f.endswith('.parquet'))
        test_df = test_df[test_df['spectrogram_id'].isin(existing_specs)].reset_index(drop=True)

        if len(test_df) > 0:
            test_dataset = HMSDataset(test_df, DATA_PATH, mode='train', augment=False) # No Augment for Test!
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

            total_test_loss = 0
            with torch.no_grad():
                for imgs, labels in test_loader:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    output = model(imgs)
                    loss = criterion(torch.log_softmax(output, dim=1), labels)
                    total_test_loss += loss.item()
            print(f"Final Test Loss (using best model): {total_test_loss / len(test_loader):.4f}")

if __name__ == "__main__":
    train()

Total rows in CSV: 61102
Unique patients: 1654
Unique eeg_id: 11416
Train rows: 10134
Val rows: 1282
Train unique eeg_id: 10134
Val unique eeg_id: 1282
Train dataset size: 10134
Val dataset size: 1282
Train loader batches: 317
Val loader batches: 41
Dummy output shape: torch.Size([2, 6])


/tmp/ipykernel_3897/1524983540.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Batch imgs shape: torch.Size([32, 3, 256, 256])
Batch labels shape: torch.Size([32, 6])
Batch output shape: torch.Size([32, 6])
Labels sum example: 1.0
Starting training on cuda...
Epoch 1/40 | Train Loss: 1.0910 | Val Loss: 1.0366
Saved best model at epoch 1 with val loss 1.0366
Epoch 2/40 | Train Loss: 0.9261 | Val Loss: 0.8620
Saved best model at epoch 2 with val loss 0.8620
Epoch 3/40 | Train Loss: 0.8430 | Val Loss: 0.9145
Epoch 4/40 | Train Loss: 0.8090 | Val Loss: 0.8093
Saved best model at epoch 4 with val loss 0.8093
Epoch 5/40 | Train Loss: 0.7964 | Val Loss: 0.7264
Saved best model at epoch 5 with val loss 0.7264
Epoch 6/40 | Train Loss: 0.7457 | Val Loss: 0.7859
Epoch 7/40 | Train Loss: 0.7180 | Val Loss: 0.7940
Epoch 8/40 | Train Loss: 0.7111 | Val Loss: 0.7832
Epoch 9/40 | Train Loss: 0.6834 | Val Loss: 0.7251
Saved best model at epoch 9 with val loss 0.7251
Epoch 10/40 | Train Loss: 0.6504 | Val Loss: 0.8015
Epoch 11/40 | Train Loss: 0.6610 | Val Loss: 0.8250
Epoch 12/40